# Grounding Audit

Every published keyword is compared with the record the model saw (truncated to 3,000 characters, as in notebook 1) and classified in this order: `null_token`; `exact_original` (a visible author or index keyword); `verbatim_text` (title or abstract); `verbatim_metadata` (authors, year or source only); `soft_original` (cosine >= 0.70 with a visible original keyword); `soft_text` (cosine >= 0.70 with a sentence of title or abstract); `ungrounded`. Records whose keyword field was cut by the truncation (`partially_visible`, `not_visible`) serve as a control (Section IV-A5, Table 5; also Sections III-B, V-A and V-B, and the coverage counts of IV-B1).

Inputs: `EID_KEYWORDS.xlsx`, `data/insumo_row_to_eid.csv`, and the private `corpus_insumo_DEFINITIVO.csv` from `FTTS_PRIVATE_DIR` (default: the parent directory of the repository). Outputs in `results/e5_grounding_leakage/`: aggregates and per-document scores keyed by EID, no record text. Gate: row alignment between records and published outputs (`validation.json`). Keywords whose best cosine sits at the 0.70 boundary can change category on other hardware. Runtime: one to two hours on CPU.

In [1]:
# ============================================================
# CONFIGURATION AND IMPORTS
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "4"        # several notebooks run concurrently on this machine
os.environ["TQDM_MININTERVAL"] = "60"      # one progress line per minute instead of a stream of carriage returns
os.environ["HF_HUB_OFFLINE"] = "1"         # embedding model read from the local Hugging Face cache, nothing is downloaded
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import sys
sys.path.insert(0, "scripts")

import json
import re
from collections import Counter
from pathlib import Path

import warnings

import numpy as np
import pandas as pd
import torch
from tqdm import TqdmWarning

torch.set_num_threads(4)
THREADS = 4
warnings.filterwarnings("ignore", category=TqdmWarning)   # no ipywidgets in this kernel: plain-text progress output

import common as C
from inspec_evaluation import normalize_phrase

OUT = C.RESULTS / "e5_grounding_leakage"
OUT.mkdir(parents=True, exist_ok=True)
CATEGORIES = ["null_token", "exact_original", "verbatim_text", "verbatim_metadata",
              "soft_original", "soft_text", "ungrounded"]
GENERIC_TERMS = {"article", "study", "analysis", "research", "work", "keyword", "keywords", "paper"}
SPANISH_STOP = {"de", "la", "el", "en", "y", "para", "del", "las", "los", "con", "una", "un", "por"}
LIMIT = None          # debug: process only the first N linked records (None = all)

INPUTS = {
    "insumo": C.PRIVATE_DIR / "corpus_insumo_DEFINITIVO.csv",
    "keywords": C.REPO_ROOT / "EID_KEYWORDS.xlsx",
    "alignment": C.DATA / "insumo_row_to_eid.csv",
}
missing = [k for k, p in INPUTS.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"missing inputs {missing}; the private records are read from FTTS_PRIVATE_DIR={C.PRIVATE_DIR}")

C.set_seeds()
T = C.Timer()
meta = C.env_metadata(experiment="E5 grounding and leakage",
                      inputs={"insumo": {"path": str(INPUTS["insumo"]), "sha256": C.sha256(INPUTS["insumo"]), "redistributed": False},
                              "keywords": {"path": str(INPUTS["keywords"]), "sha256": C.sha256(INPUTS["keywords"])},
                              "alignment": {"path": str(INPUTS["alignment"]), "sha256": C.sha256(INPUTS["alignment"])}},
                      llm_calls=0, paid_api_calls=0, truncate_chars=C.TRUNCATE_CHARS, tau_soft=C.TAU_SOFT)
print(json.dumps({k: v for k, v in meta.items() if k != "inputs"}, indent=1))
print(pd.DataFrame([{"input": k, **v} for k, v in meta["inputs"].items()]).to_string(index=False))
print("results dir:", OUT, "| truncate_chars:", C.TRUNCATE_CHARS, "| tau_soft:", C.TAU_SOFT, "| seed:", C.SEED)

{
 "started_utc": "2026-09-25T14:58:31.897300+00:00",
 "python": "3.12.3",
 "platform": "Linux-7.0.0-31-generic-x86_64-with-glibc2.39",
 "machine": "x86_64",
 "cpu_count": 16,
 "git_commit": "af664fad8d65723998851025c9d052e9bf1fa73e",
 "packages": {
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "scipy": "1.15.3",
  "sklearn": "1.9.0",
  "networkx": "3.6.1",
  "community": "0.16",
  "sentence_transformers": "5.1.1",
  "torch": "2.8.0+cpu",
  "transformers": "4.57.6",
  "langdetect": "unknown",
  "openai": "1.107.2"
 },
 "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
 "embedding_model_revision": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
 "experiment": "E5 grounding and leakage",
 "llm_calls": 0,
 "paid_api_calls": 0,
 "truncate_chars": 3000,
 "tau_soft": 0.7
}
    input                                                                                          path                                                           sha256 redistributed
   insumo     

In [2]:
# ============================================================
# HELPERS: normalised substring test and visibility of the original keywords after truncation
# ============================================================
def norm_text(s: str) -> str:
    """Same normalisation as phrases, applied to running text, so substring tests are consistent."""
    return normalize_phrase(s)


def contains_phrase(text_norm: str, phrase: str) -> bool:
    if not phrase:
        return False
    return re.search(r"(?<![a-z0-9])" + re.escape(phrase) + r"(?![a-z0-9])", text_norm) is not None


def visible_original_keywords(rec: dict, insumo: str) -> tuple[list[str], str]:
    """Original keywords whose text lies inside the first TRUNCATE_CHARS characters."""
    start = rec["keyword_field_start"]
    if start >= C.TRUNCATE_CHARS:
        return [], "not_visible"
    seen_raw = insumo[start:C.TRUNCATE_CHARS]
    full_raw = insumo[start:]
    status = "fully_visible" if len(insumo) <= C.TRUNCATE_CHARS else "partially_visible"
    # Keep only keywords completely inside the visible span (a cut keyword was not seen whole).
    kws = []
    for k in seen_raw.split(";"):
        t = normalize_phrase(k)
        if t:
            kws.append(t)
    if status == "partially_visible" and kws and not full_raw.rstrip().endswith(seen_raw.rstrip()):
        # The last visible fragment may be a truncated keyword: drop it unless it also occurs in full.
        full_set = set(C.split_original_keywords(full_raw))
        if kws[-1] not in full_set:
            kws = kws[:-1]
    return list(dict.fromkeys(kws)), status


# Sanity check of the substring test: whole-word match after normalisation, no partial-word match.
print("'mathematics education' in 'Mathematics Education in Chile.':", contains_phrase(norm_text("Mathematics Education in Chile."), "mathematics education"))
print("'statistic' in 'statistics education':", contains_phrase(norm_text("statistics education"), "statistic"))

'mathematics education' in 'Mathematics Education in Chile.': True
'statistic' in 'statistics education': False


In [3]:
# ============================================================
# LOAD, ALIGN AND GATE (row alignment between the private records and the published keywords)
# ============================================================
ins_all = pd.read_csv(INPUTS["insumo"])["insumo"].astype(str).tolist()
align_df = C.load_alignment(INPUTS["alignment"])
pub = C.load_published_keywords(INPUTS["keywords"], notebook_semantics=False)
if LIMIT:
    align_df = align_df.head(LIMIT)
# Work on the linked records only: ``ins[i]`` is the record of published output ``kw.keywords[i]``.
ins = [ins_all[r] for r in align_df.insumo_row]
kw = pd.DataFrame({"eid": align_df.eid.values, "keywords": [pub[e] for e in align_df.eid]})
n = len(kw)
print(f"record rows (extraction run) = {len(ins_all):,} | published outputs in EID_KEYWORDS.xlsx = {len(pub):,} "
      f"| published outputs linked to a record = {n:,}")
print(f"keywords in the published file = {sum(len(v) for v in pub.values()):,} | keywords audited here = {sum(len(v) for v in kw.keywords):,} "
      f"| theoretical maximum of the extraction run = {5 * len(ins_all):,}")


# Alignment gate: share of generated keywords found verbatim in the same row versus shifted rows.
def literal_share(offset, rows=3000):
    vals = []
    for i in range(0, min(rows, n)):
        j = i + offset
        if j < 0 or j >= len(ins):
            continue
        kws = [normalize_phrase(k) for k in kw["keywords"][i]]
        kws = [k for k in kws if k and k != "null"]
        if kws:
            t = norm_text(ins[j])
            vals.append(np.mean([contains_phrase(t, k) for k in kws]))
    return float(np.mean(vals)) if vals else float("nan")


align = {"offset_0": literal_share(0), "offset_plus1": literal_share(1), "offset_minus1": literal_share(-1)}
align_pass = align["offset_0"] > 0.6 and align["offset_0"] - max(align["offset_plus1"], align["offset_minus1"]) > 0.3
print("ALIGNMENT GATE:", "PASS" if align_pass else "FAIL", {k: round(v, 4) for k, v in align.items()})
unmatched_tail = {"record_rows_total": len(ins_all), "records_analyzed": n,
                  "records_not_linked_to_a_published_output": len(ins_all) - n}
print(unmatched_tail)
print(f"elapsed {T.mark('load'):.1f} s")

record rows (extraction run) = 53,130 | published outputs in EID_KEYWORDS.xlsx = 52,947 | published outputs linked to a record = 52,947
keywords in the published file = 264,586 | keywords audited here = 264,586 | theoretical maximum of the extraction run = 265,650


ALIGNMENT GATE: PASS {'offset_0': 0.8452, 'offset_plus1': 0.1128, 'offset_minus1': 0.1153}
{'record_rows_total': 53130, 'records_analyzed': 52947, 'records_not_linked_to_a_published_output': 183}
elapsed 9.8 s


In [4]:
# ============================================================
# LEXICAL PASS: null token, exact copy of a visible original keyword, verbatim in text or metadata
# ============================================================
recs, gen_rows = [], []
all_gen, all_orig, all_sent = set(), set(), []
doc_sentences = []
for i in range(n):
    s = ins[i]
    seen = s[:C.TRUNCATE_CHARS]
    rec = C.parse_record(s)
    rec_seen = C.parse_record(seen)          # fields as the model saw them
    orig_seen, vis = visible_original_keywords(rec, s)
    orig_full = C.split_original_keywords(rec["original_keywords_raw"])
    text_seen_norm = norm_text(rec_seen["title"] + " . " + rec_seen["abstract"])
    meta_norm = norm_text(rec_seen["authors"] + " " + rec_seen["year"] + " " + rec_seen["source"])
    sents = C.split_sentences(rec_seen["title"] + ". " + rec_seen["abstract"])
    doc_sentences.append(sents)
    all_sent.extend(sents)
    all_orig.update(orig_seen)
    gen_raw = kw["keywords"][i]
    gen = []
    for g in gen_raw:
        t = normalize_phrase(g)
        gen.append(t)
    recs.append({"row": i, "eid": kw["eid"][i], "visibility": vis, "record_length": rec["length"],
                 "n_generated": len(gen_raw), "n_original_full": len(orig_full), "n_original_seen": len(orig_seen),
                 "orig_seen": orig_seen, "orig_full": orig_full})
    for pos, (raw, t) in enumerate(zip(gen_raw, gen)):
        if not t or t == "null":
            cat = "null_token"
        elif t in set(orig_seen):
            cat = "exact_original"
        elif contains_phrase(text_seen_norm, t):
            cat = "verbatim_text"
        elif contains_phrase(meta_norm, t):
            cat = "verbatim_metadata"
        else:
            cat = None                          # decided semantically below
        gen_rows.append({"row": i, "eid": kw["eid"][i], "position": pos, "keyword": t, "raw": str(raw),
                         "lexical_category": cat,
                         "in_original_seen": bool(t) and t != "null" and t in set(orig_seen),
                         "in_text": bool(t) and t != "null" and contains_phrase(text_seen_norm, t), "in_original_full_not_seen": (t in set(orig_full)) and (t not in set(orig_seen)),
                         "n_words": len(t.split()), "non_ascii": any(ord(ch) > 127 for ch in str(raw)),
                         "generic_term": t in GENERIC_TERMS, "spanish_stopword": any(w in SPANISH_STOP for w in t.split())})
        if t and t != "null":
            all_gen.add(t)
    if i % 10000 == 0:
        print(f"  lexical pass {i}/{n}", flush=True)
G = pd.DataFrame(gen_rows)
D = pd.DataFrame(recs)
T.mark("lexical")
print(f"\nkeywords = {len(G):,} | documents = {len(D):,} | distinct generated keywords = {len(all_gen):,} "
      f"| distinct visible original keywords = {len(all_orig):,}")
print("lexical categories (None = pending semantic check):")
print(G.lexical_category.fillna("pending").value_counts().to_string())
print("\nvisibility of the original keyword field in the truncated record:")
print(D.visibility.value_counts().to_string())

  lexical pass 0/52947


  lexical pass 10000/52947


  lexical pass 20000/52947


  lexical pass 30000/52947


  lexical pass 40000/52947


  lexical pass 50000/52947



keywords = 264,586 | documents = 52,947 | distinct generated keywords = 55,855 | distinct visible original keywords = 93,055
lexical categories (None = pending semantic check):
lexical_category
exact_original       119748
verbatim_text        106300
pending               36862
verbatim_metadata      1484
null_token              192

visibility of the original keyword field in the truncated record:
visibility
fully_visible        50939
not_visible           1173
partially_visible      835


In [5]:
# ============================================================
# SEMANTIC PASS 1: embed the pending generated keywords and the visible original keywords
# ============================================================
model = C.load_embedder(threads=THREADS)
pending = G[G.lexical_category.isna()]
print(f"keywords needing semantic check: {len(pending):,} of {len(G):,} ({len(pending) / len(G):.1%})", flush=True)
emb_gen = C.embed_unique(model, set(pending.keyword), batch_size=256, desc="generated keywords")
emb_orig = C.embed_unique(model, all_orig, batch_size=256, desc="original keywords")
T.mark("embed_phrases")
print(f"embedded {len(emb_gen):,} distinct pending keywords and {len(emb_orig):,} distinct original keywords "
      f"| elapsed {T.marks['embed_phrases']:.0f} s")

keywords needing semantic check: 36,862 of 264,586 (13.9%)


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

Batches: 100%|██████████| 37/37 [00:20<00:00,  1.81it/s]

Batches:   0%|          | 0/364 [00:00<?, ?it/s]

Batches:  21%|██        | 75/364 [01:00<03:53,  1.24it/s]

...

Batches:  49%|████▉     | 179/364 [02:19<02:01,  1.52it/s]

Batches:  81%|████████▏ | 296/364 [03:01<00:39,  1.72it/s]

Batches:  81%|████████▏ | 296/364 [03:19<00:39,  1.72it/s]

Batches: 100%|██████████| 364/364 [03:29<00:00,  1.73it/s]

embedded 9,384 distinct pending keywords and 93,055 distinct original keywords | elapsed 278 s


In [6]:
# ============================================================
# SEMANTIC PASS 2: embed the sentences of title + abstract (only for documents with pending keywords)
# ============================================================
need_docs = set(pending.row)
sent_list = sorted(set(s for i in need_docs for s in doc_sentences[i]))
print(f"sentences to embed: {len(sent_list):,} from {len(need_docs):,} documents", flush=True)
emb_sent = C.embed_unique(model, sent_list, batch_size=256, desc="sentences")
T.mark("embed_sentences")
print(f"elapsed {T.marks['embed_sentences']:.0f} s")

sentences to embed: 205,673 from 25,143 documents


Batches:   0%|          | 0/804 [00:00<?, ?it/s]

Batches:   1%|          | 7/804 [01:08<2:10:28,  9.82s/it]

Batches:   1%|          | 7/804 [01:26<2:10:28,  9.82s/it]

Batches:   2%|▏         | 13/804 [02:10<2:12:33, 10.05s/it]

...

Batches:  91%|█████████▏| 734/804 [44:06<02:08,  1.83s/it]

Batches:  97%|█████████▋| 779/804 [44:53<00:41,  1.65s/it]

Batches:  97%|█████████▋| 779/804 [45:06<00:41,  1.65s/it]

Batches: 100%|██████████| 804/804 [45:14<00:00,  3.38s/it]

elapsed 2995 s


In [7]:
# ============================================================
# SEMANTIC PASS 3: soft_original (cos >= tau with a visible original keyword), soft_text (cos >= tau with a sentence), else ungrounded
# ============================================================
max_sim_orig = np.full(len(G), np.nan)
max_sim_sent = np.full(len(G), np.nan)
final = G.lexical_category.copy()
for idx, r in pending.iterrows():
    v = emb_gen[r.keyword]
    orig = recs[r.row]["orig_seen"]
    so = max((float(v @ emb_orig[o]) for o in orig if o in emb_orig), default=np.nan)
    sents = doc_sentences[r.row]
    ss = max((float(v @ emb_sent[s]) for s in sents if s in emb_sent), default=np.nan)
    max_sim_orig[idx], max_sim_sent[idx] = so, ss
    if not np.isnan(so) and so >= C.TAU_SOFT:
        final[idx] = "soft_original"
    elif not np.isnan(ss) and ss >= C.TAU_SOFT:
        final[idx] = "soft_text"
    else:
        final[idx] = "ungrounded"
G["category"] = final
G["max_sim_original_seen"] = max_sim_orig
G["max_sim_sentence"] = max_sim_sent
T.mark("semantic")
print("final categories:")
print(G.category.value_counts().reindex(CATEGORIES).to_string())
near = G[G.category.isin(["soft_original", "soft_text", "ungrounded"])]
best = near[["max_sim_original_seen", "max_sim_sentence"]].max(axis=1)
print(f"\nkeywords whose best similarity lies within 0.005 of tau = {C.TAU_SOFT}: {int((best.sub(C.TAU_SOFT).abs() < 0.005).sum()):,} "
      f"(these are the ones that could change category on other hardware)")

final categories:
category
null_token              192
exact_original       119748
verbatim_text        106300
verbatim_metadata      1484
soft_original         19206
soft_text              1093
ungrounded            16563

keywords whose best similarity lies within 0.005 of tau = 0.7: 673 (these are the ones that could change category on other hardware)


In [8]:
# ============================================================
# DOCUMENT-LEVEL GLOBAL SIMILARITY (Table 2 metric: concatenated keywords vs title + abstract, 128-token embedder limit)
# ============================================================
doc_texts = [C.parse_record(ins[i][:C.TRUNCATE_CHARS]) for i in range(n)]
doc_strings = [(d["title"] + ". " + d["abstract"]).strip() for d in doc_texts]
kw_strings = [" ; ".join(k for k in G[G.row == i].keyword if k and k != "null") for i in range(n)]
E_doc = model.encode(doc_strings, batch_size=128, normalize_embeddings=True, show_progress_bar=True)
E_kw = model.encode(kw_strings, batch_size=256, normalize_embeddings=True, show_progress_bar=True)
D["global_sem_sim"] = np.einsum("ij,ij->i", E_doc, E_kw)
T.mark("doc_similarity")
print(f"global similarity: mean = {D.global_sem_sim.mean():.4f} | sd = {D.global_sem_sim.std(ddof=1):.4f} "
      f"| median = {D.global_sem_sim.median():.4f} | elapsed {T.marks['doc_similarity']:.0f} s")

Batches:   0%|          | 0/414 [00:00<?, ?it/s]

Batches:   4%|▍         | 17/414 [01:03<24:32,  3.71s/it]

Batches:   4%|▍         | 17/414 [01:21<24:32,  3.71s/it]

Batches:   8%|▊         | 33/414 [02:03<23:44,  3.74s/it]

Batches:   8%|▊         | 33/414 [02:21<23:44,  3.74s/it]

...

Batches:  68%|██████▊   | 141/207 [04:20<01:51,  1.69s/it]

Batches:  88%|████████▊ | 183/207 [05:06<00:38,  1.60s/it]

Batches:  88%|████████▊ | 183/207 [05:20<00:38,  1.60s/it]

Batches: 100%|██████████| 207/207 [05:35<00:00,  1.62s/it]

global similarity: mean = 0.7188 | sd = 0.0917 | median = 0.7313 | elapsed 4871 s


In [9]:
# ============================================================
# AGGREGATES: per document, category shares, visibility strata, ungrounded keywords, similarity histograms
# ============================================================
G["is_leak_exact"] = G.category.eq("exact_original")
G["is_leak_any"] = G.category.isin(["exact_original", "soft_original"])
G["is_grounded"] = ~G.category.isin(["ungrounded", "null_token"])
per_doc = G.groupby("row").agg(n_keywords=("keyword", "size"),
                               n_null=("category", lambda c: (c == "null_token").sum()),
                               n_exact_original=("is_leak_exact", "sum"),
                               n_leak_any=("is_leak_any", "sum"),
                               n_verbatim_text=("category", lambda c: (c == "verbatim_text").sum()),
                               n_ungrounded=("category", lambda c: (c == "ungrounded").sum()),
                               n_unique=("keyword", "nunique"))
D = D.join(per_doc, on="row")
D["share_exact_original"] = D.n_exact_original / D.n_keywords.clip(lower=1)
D["share_leak_any"] = D.n_leak_any / D.n_keywords.clip(lower=1)


# Reverse direction: how many of the visible original keywords were reproduced exactly.
def reproduced(row):
    s = set(G[(G.row == row.row)].keyword)
    return sum(1 for o in row.orig_seen if o in s)


D["n_original_seen_reproduced"] = [reproduced(r) for r in D.itertuples()]
D["recall_original_seen"] = np.where(D.n_original_seen > 0, D.n_original_seen_reproduced / D.n_original_seen.clip(lower=1), np.nan)

cat_share = G.category.value_counts(normalize=True).reindex(CATEGORIES).fillna(0)
cat_count = G.category.value_counts().reindex(CATEGORIES).fillna(0).astype(int)
shares = pd.DataFrame({"category": CATEGORIES, "keywords": cat_count.values, "share": cat_share.values})
shares.to_csv(OUT / "category_shares.csv", index=False)

strat = (G.merge(D[["row", "visibility"]], on="row")
         .groupby(["visibility", "category"]).size().unstack(fill_value=0).reindex(columns=CATEGORIES, fill_value=0))
strat_share = strat.div(strat.sum(axis=1), axis=0)
strat_share["n_keywords"] = strat.sum(axis=1)
strat_share["n_documents"] = D.groupby("visibility").size()
strat_share["mean_global_sem_sim"] = D.groupby("visibility").global_sem_sim.mean()
strat_share["sd_global_sem_sim"] = D.groupby("visibility").global_sem_sim.std(ddof=1)
strat_share["mean_record_length"] = D.groupby("visibility").record_length.mean()
strat_share["mean_share_exact_original"] = D.groupby("visibility").share_exact_original.mean()
strat_share["mean_recall_original_seen"] = D.groupby("visibility").recall_original_seen.mean()
strat_share.reset_index().to_csv(OUT / "by_visibility_stratum.csv", index=False)

# Ungrounded keywords: most frequent and a random sample for qualitative inspection.
ung = G[G.category == "ungrounded"]
ung_top = ung.keyword.value_counts().head(50).rename_axis("keyword").reset_index(name="count")
ung_top.to_csv(OUT / "ungrounded_top50.csv", index=False)
ung.sample(min(150, len(ung)), random_state=C.SEED)[["eid", "keyword", "max_sim_original_seen", "max_sim_sentence"]].to_csv(
    OUT / "ungrounded_sample150.csv", index=False)
# Distribution of the best sentence similarity for ungrounded vs soft_text keywords.
bins = np.round(np.arange(0, 1.0001, 0.05), 2)
for cat in ("ungrounded", "soft_text", "soft_original"):
    sub = G[G.category == cat]
    col = "max_sim_original_seen" if cat == "soft_original" else "max_sim_sentence"
    h = pd.cut(sub[col], bins, include_lowest=True).value_counts().sort_index()
    h.rename_axis("bin").reset_index(name="count").to_csv(OUT / f"similarity_hist_{cat}.csv", index=False)

pd.set_option("display.width", 250)
print("CATEGORY SHARES (all audited keywords)")
print(shares.assign(share_pct=(shares.share * 100).round(2)).to_string(index=False))
print("\nBY VISIBILITY OF THE ORIGINAL KEYWORDS IN THE TRUNCATED RECORD (shares within stratum)")
print(strat_share[CATEGORIES + ["n_keywords", "n_documents", "mean_global_sem_sim", "mean_record_length",
                                "mean_share_exact_original", "mean_recall_original_seen"]].round(4).T.to_string())
print("\nMOST FREQUENT UNGROUNDED KEYWORDS")
print(ung_top.head(15).to_string(index=False))
top3 = ung_top["count"].head(3).sum()
print(f"\nthree most frequent ungrounded terms: {top3:,} of {len(ung):,} ungrounded keywords ({top3 / max(len(ung), 1):.1%})")
print(f"best-sentence similarity of ungrounded keywords: mean = {ung.max_sim_sentence.mean():.4f}, median = {ung.max_sim_sentence.median():.4f}")

CATEGORY SHARES (all audited keywords)
         category  keywords    share  share_pct
       null_token       192 0.000726       0.07
   exact_original    119748 0.452586      45.26
    verbatim_text    106300 0.401760      40.18
verbatim_metadata      1484 0.005609       0.56
    soft_original     19206 0.072589       7.26
        soft_text      1093 0.004131       0.41
       ungrounded     16563 0.062600       6.26

BY VISIBILITY OF THE ORIGINAL KEYWORDS IN THE TRUNCATED RECORD (shares within stratum)
visibility                 fully_visible  not_visible  partially_visible
category                                                                
null_token                        0.0008       0.0000             0.0000
exact_original                    0.4640       0.0000             0.3908
verbatim_text                     0.3916       0.7922             0.4740
verbatim_metadata                 0.0058       0.0024             0.0010
soft_original                     0.0746       0.00

In [10]:
# ============================================================
# PROMPT ADHERENCE, 2x2 ATTRIBUTION, SUMMARY AND OUTPUT FILES
# ============================================================
adherence = {
    "documents": int(n),
    "keywords_total": int(len(G)),
    "documents_with_exactly_5_keywords": int((D.n_generated == 5).sum()),
    "documents_with_fewer_than_5": int((D.n_generated < 5).sum()),
    "documents_with_more_than_5": int((D.n_generated > 5).sum()),
    "null_tokens": int(cat_count["null_token"]),
    "null_token_rate": float(cat_count["null_token"] / len(G)),
    "documents_with_duplicate_keywords": int((D.n_unique < D.n_generated).sum()),
    "keywords_with_non_ascii_characters": int(G.non_ascii.sum()),
    "keywords_equal_to_forbidden_generic_terms": int(G.generic_term.sum()),
    "keywords_longer_than_4_words": int((G.n_words > 4).sum()),
    "keywords_with_spanish_function_words": int(G.spanish_stopword.sum()),
    "examples_spanish_function_words": G[G.spanish_stopword].keyword.value_counts().head(15).to_dict(),
}

valid = G[G.category != "null_token"]
attribution = {
    "in_original_only": float((valid.in_original_seen & ~valid.in_text).mean()),
    "in_original_and_text": float((valid.in_original_seen & valid.in_text).mean()),
    "in_text_only": float((~valid.in_original_seen & valid.in_text).mean()),
    "in_neither": float((~valid.in_original_seen & ~valid.in_text).mean()),
    "keywords": int(len(valid)),
    "note": "exact lexical presence in the record the model saw; categories above are hierarchical, this table is not",
}
pd.DataFrame([attribution]).to_csv(OUT / "attribution_2x2.csv", index=False)
summary = {
    "documents_analyzed": int(n),
    "attribution_2x2_exact_presence": attribution,
    "keywords_classified": int(len(G)),
    "category_shares": shares.set_index("category")["share"].round(5).to_dict(),
    "category_counts": shares.set_index("category")["keywords"].to_dict(),
    "leakage": {
        "share_keywords_exact_copy_of_visible_original": float(G.is_leak_exact.mean()),
        "share_keywords_exact_or_paraphrase_of_visible_original": float(G.is_leak_any.mean()),
        "share_keywords_matching_original_hidden_by_truncation": float(G.in_original_full_not_seen.mean()),
        "mean_per_document_share_exact_original": float(D.share_exact_original.mean()),
        "median_per_document_share_exact_original": float(D.share_exact_original.median()),
        "documents_with_zero_exact_copies": int((D.n_exact_original == 0).sum()),
        "documents_with_all_keywords_exact_copies": int((D.n_exact_original == D.n_keywords).sum()),
        "mean_recall_of_visible_original_keywords": float(D.recall_original_seen.mean()),
        "mean_n_original_keywords_visible": float(D.n_original_seen.mean()),
    },
    "grounding": {
        "share_grounded_any_level": float(G.is_grounded.mean()),
        "share_verbatim_in_record": float(G.category.isin(["exact_original", "verbatim_text", "verbatim_metadata"]).mean()),
        "share_ungrounded": float(G.category.eq("ungrounded").mean()),
        "documents_with_at_least_one_ungrounded": int((D.n_ungrounded > 0).sum()),
        "mean_max_sentence_similarity_ungrounded": float(ung.max_sim_sentence.mean()),
        "median_max_sentence_similarity_ungrounded": float(ung.max_sim_sentence.median()),
    },
    "visibility_of_original_keywords": D.visibility.value_counts().to_dict(),
    "by_visibility_stratum": strat_share.round(5).reset_index().to_dict(orient="records"),
    "prompt_adherence": adherence,
    "alignment": {**align, "pass": bool(align_pass), **unmatched_tail},
}
C.write_json(summary, OUT / "summary.json")
D.drop(columns=["orig_seen", "orig_full"]).to_csv(OUT / "per_document.csv", index=False)
G[["eid", "position", "keyword", "category", "max_sim_original_seen", "max_sim_sentence"]].to_csv(
    OUT / "per_keyword.csv.gz", index=False, compression="gzip")
C.write_json({"gate": "row alignment insumo <-> keyword workbook", "pass": bool(align_pass), **align,
              **unmatched_tail}, OUT / "validation.json")
meta.update({"completed_utc": C.now_utc(), "timings_seconds": T.marks, "status": "complete"})
C.write_json(meta, OUT / "metadata.json")

print("PROMPT ADHERENCE")
print(json.dumps(adherence, indent=1, ensure_ascii=False))
print("\n2x2 ATTRIBUTION (exact lexical presence, non-null keywords)")
print(pd.DataFrame([attribution]).drop(columns="note").T.to_string(header=False))
print("\nLEAKAGE")
print(json.dumps(summary["leakage"], indent=1))
print("\nGROUNDING")
print(json.dumps(summary["grounding"], indent=1))
print("\ntimings (s):", T.marks)
print("files written:", sorted(p.name for p in OUT.iterdir()))

PROMPT ADHERENCE
{
 "documents": 52947,
 "keywords_total": 264586,
 "documents_with_exactly_5_keywords": 52740,
 "documents_with_fewer_than_5": 183,
 "documents_with_more_than_5": 24,
 "null_tokens": 192,
 "null_token_rate": 0.0007256619775800685,
 "documents_with_duplicate_keywords": 0,
 "keywords_with_non_ascii_characters": 38,
 "keywords_equal_to_forbidden_generic_terms": 276,
 "keywords_longer_than_4_words": 294,
 "keywords_with_spanish_function_words": 24,
 "examples_spanish_function_words": {
  "augustus de morgan": 2,
  "rio de janeiro": 2,
  "generation y": 1,
  "universidad politecnica de valencia": 1,
  "south central los angeles": 1,
  "accademia del cimento": 1,
  "g del s incompleteness theorem": 1,
  "el hi textbooks": 1,
  "de facto standards": 1,
  "las vegas": 1,
  "de colonization": 1,
  "de morgan": 1,
  "olympic para alpine skiers": 1,
  "para athletes": 1,
  "de marginalization": 1
 }
}

2x2 ATTRIBUTION (exact lexical presence, non-null keywords)
in_original_only  

## Check against the manuscript

In [11]:
# ============================================================
# CHECK AGAINST THE MANUSCRIPT (main.tex values transcribed by hand; nothing here edits the manuscript)
# ============================================================
def manuscript_table(rows):
    out = []
    for loc, qty, tex, val, nd in rows:
        try:
            tex_num = float(str(tex).replace(",", "").replace("%", ""))
        except ValueError:
            tex_num = None
        if val is None or (isinstance(val, float) and np.isnan(val)):
            comp, flag = "nan", "n/a"
        else:
            comp = f"{float(val):,.{nd}f}" if nd > 0 else f"{int(round(float(val))):,}"
            if tex_num is None:
                flag = "n/a (not in main.tex)"
            else:
                flag = "match" if abs(round(float(val), nd) - tex_num) < 1e-9 else "differs"
        out.append({"location": loc, "quantity": qty, "main.tex": str(tex), "computed": comp, "flag": flag})
    return pd.DataFrame(out)


pct = lambda x: 100 * float(x)
sh = summary["category_shares"]
att = attribution
lk, gr = summary["leakage"], summary["grounding"]
st = {r["visibility"]: r for r in summary["by_visibility_stratum"]}
S4 = "Sec. IV-A5 text"
TB = "Table 5 (tab:grounding)"
S3 = "Sec. III-B, hallucination criterion"
S5A = "Sec. V-A Interpretability"
S5B = "Sec. V-B Limitations"
S4B = "Sec. IV-B1 Coverage"
S4C = "Sec. IV-B1 Coverage"
LET = "Response letter (R1 / Editor)"
names = {"exact_original": "Exact copy of a record keyword", "verbatim_text": "Verbatim in title or abstract",
         "verbatim_metadata": "Verbatim in other fields", "soft_original": "Paraphrase of a record keyword (cos >= 0.70)",
         "soft_text": "Close to a sentence (cos >= 0.70)", "ungrounded": "Ungrounded", "null_token": "null placeholder"}
# Table 5 cells, columns: visible / partly / hidden (percent). Transcribed row by row from main.tex.
table5 = {
    "exact_original": {"fully_visible": "46.4", "partially_visible": "39.1", "not_visible": "0.0"},
    "verbatim_text": {"fully_visible": "39.2", "partially_visible": "47.4", "not_visible": "79.2"},
    "verbatim_metadata": {"fully_visible": "0.6", "partially_visible": "0.1", "not_visible": "0.2"},
    "soft_original": {"fully_visible": "7.5", "partially_visible": "5.5", "not_visible": "0.0"},
    "soft_text": {"fully_visible": "0.4", "partially_visible": "0.5", "not_visible": "1.9"},
    "ungrounded": {"fully_visible": "6.0", "partially_visible": "7.5", "not_visible": "18.7"},
    "null_token": {"fully_visible": "0.07", "partially_visible": "0.0", "not_visible": "0.0"},
}
col = {"fully_visible": "visible", "partially_visible": "partly", "not_visible": "hidden"}
vis_mean = st["fully_visible"]["mean_global_sem_sim"]
hid_mean = st["not_visible"]["mean_global_sem_sim"]

rows = [
    (S4, "keywords compared with the record", "264,586", len(G), 0),
    (S4, "documents (Table 5, All records)", "52,947", n, 0),
    (S4, "exact copy of a visible record keyword (%)", "45.3", pct(sh["exact_original"]), 1),
    (S4, "verbatim in title or abstract (%)", "40.2", pct(sh["verbatim_text"]), 1),
    (S4, "verbatim in the remaining metadata fields (%)", "0.6", pct(sh["verbatim_metadata"]), 1),
    (S4, "paraphrase of a record keyword, cos >= 0.70 (%)", "7.3", pct(sh["soft_original"]), 1),
    (S4, "close to a sentence of title or abstract, cos >= 0.70 (%)", "0.4", pct(sh["soft_text"]), 1),
    (S4, "ungrounded (%)", "6.3", pct(sh["ungrounded"]), 1),
    (S4, "null placeholder (%)", "0.07", pct(sh["null_token"]), 2),
    (S4, "unordered: occur among the record keywords (%)", "45.3", pct(att["in_original_only"] + att["in_original_and_text"]), 1),
    (S4, "unordered: occur in the text (%)", "72.6", pct(att["in_text_only"] + att["in_original_and_text"]), 1),
    (S4, "unordered: in both (%)", "32.4", pct(att["in_original_and_text"]), 1),
    (S4, "unordered: only among the record keywords (%)", "12.9", pct(att["in_original_only"]), 1),
    (S4, "unordered: in neither (%)", "14.5", pct(att["in_neither"]), 1),
    (TB, "Documents, visible", "50,939", st["fully_visible"]["n_documents"], 0),
    (TB, "Documents, partly", "835", st["partially_visible"]["n_documents"], 0),
    (TB, "Documents, hidden", "1,173", st["not_visible"]["n_documents"], 0),
]
for cat in ["exact_original", "verbatim_text", "verbatim_metadata", "soft_original", "soft_text", "ungrounded", "null_token"]:
    for strat_key in ["fully_visible", "partially_visible", "not_visible"]:
        tex = table5[cat][strat_key]
        rows.append((TB, f"{names[cat]}, {col[strat_key]} (%)", tex, pct(st[strat_key][cat]), len(tex.split(".")[1])))
rows += [
    (TB, "Global similarity keywords vs document, All", "0.719", D.global_sem_sim.mean(), 3),
    (TB, "Global similarity, visible", "0.720", vis_mean, 3),
    (TB, "Global similarity, partly", "0.703", st["partially_visible"]["mean_global_sem_sim"], 3),
    (TB, "Global similarity, hidden", "0.698", hid_mean, 3),
    (S4, "ungrounded keywords (count)", "16,563", summary["category_counts"]["ungrounded"], 0),
    (S4, "share of ungrounded held by the three most frequent terms (%)", "36.4", pct(top3 / max(len(ung), 1)), 1),
    (S4, "median best-sentence similarity of an ungrounded keyword", "0.52", gr["median_max_sentence_similarity_ungrounded"], 2),
    (S4, "median document: keywords copied from the record keyword field (of 5)", "2", lk["median_per_document_share_exact_original"] * 5, 0),
    (S4, "documents with all five keywords copied", "4,817", lk["documents_with_all_keywords_exact_copies"], 0),
    (S4, "documents with all five keywords copied (%)", "9.1", pct(lk["documents_with_all_keywords_exact_copies"] / n), 1),
    (S4, "documents with no keyword copied", "10,363", lk["documents_with_zero_exact_copies"], 0),
    (S4, "documents with no keyword copied (%)", "19.6", pct(lk["documents_with_zero_exact_copies"] / n), 1),
    (S4, "visible record keywords reproduced by the model (%)", "30.9", pct(lk["mean_recall_of_visible_original_keywords"]), 1),
    (S4, "visible record keywords per record (mean)", "9.5", lk["mean_n_original_keywords_visible"], 1),
    (S4, "records with the keyword field hidden by the truncation", "1,173", summary["visibility_of_original_keywords"].get("not_visible", 0), 0),
    (S4, "records with the keyword field partly visible", "835", summary["visibility_of_original_keywords"].get("partially_visible", 0), 0),
    (S4, "hidden records: verbatim from title and abstract (%)", "79.2", pct(st["not_visible"]["verbatim_text"]), 1),
    (S4, "hidden records: ungrounded (%)", "18.7", pct(st["not_visible"]["ungrounded"]), 1),
    (S4, "hidden records: global similarity", "0.698", hid_mean, 3),
    (S4, "visible records: global similarity", "0.720", vis_mean, 3),
    (S4 + " and " + S5B, "cost of hiding the record keywords in document-level similarity (at most)", "0.02", vis_mean - hid_mean, 2),
    (S3, "cosine threshold for paraphrase / sentence grounding", "0.70", C.TAU_SOFT, 2),
    (S3, "record truncation seen by the model (characters)", "3,000", C.TRUNCATE_CHARS, 0),
    (S5A, "keywords verbatim in the record (%)", "86.0", pct(gr["share_verbatim_in_record"]), 1),
    (S5A, "keywords paraphrasing record keywords or sentences (%)", "7.7", pct(sh["soft_original"] + sh["soft_text"]), 1),
    (S5A, "keywords without an anchor (%)", "6.3", pct(gr["share_ungrounded"]), 1),
    (LET, "keywords traceable to the record (%)", "93.7", pct(gr["share_grounded_any_level"]), 1),
    (S5B, "keywords copying a record keyword (%)", "45.3", pct(lk["share_keywords_exact_copy_of_visible_original"]), 1),
    (S5B, "keywords copying or paraphrasing a record keyword (%)", "52.5", pct(lk["share_keywords_exact_or_paraphrase_of_visible_original"]), 1),
    (S4B, "documents of the corpus", "52,947", n, 0),
    (S4B, "keywords generated", "264,586", len(G), 0),
    (S4B, "theoretical maximum", "264,735", 5 * n, 0),
    (S4B, "keyword-level coverage (%)", "99.94", pct(len(G) / (5 * n)), 2),
    (S4C, "documents with exactly five keywords", "52,740", adherence["documents_with_exactly_5_keywords"], 0),
    (S4C, "documents with exactly five keywords (%)", "99.6", pct(adherence["documents_with_exactly_5_keywords"] / n), 1),
    (S4C, "documents with fewer than five keywords", "183", adherence["documents_with_fewer_than_5"], 0),
    (S4C, "documents with more than five keywords", "24", adherence["documents_with_more_than_5"], 0),
    (S4C, "null placeholders", "192", adherence["null_tokens"], 0),
    (S4C, "null placeholders (%)", "0.073", pct(adherence["null_token_rate"]), 3),
    (S4C, "keywords equal to a forbidden generic term", "276", adherence["keywords_equal_to_forbidden_generic_terms"], 0),
    (S4C, "keywords equal to a forbidden generic term (%)", "0.10", pct(adherence["keywords_equal_to_forbidden_generic_terms"] / len(G)), 2),
    (S4C, "keywords with non-ASCII characters", "38", adherence["keywords_with_non_ascii_characters"], 0),
    (S4C, "keywords with a Spanish function word", "24", adherence["keywords_with_spanish_function_words"], 0),
    (S4C, "lists with a repeated keyword", "0", adherence["documents_with_duplicate_keywords"], 0),
]
check = manuscript_table(rows)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)
print(check.to_string(index=False))
print(f"\n{(check.flag == 'match').sum()} match, {(check.flag == 'differs').sum()} differ, "
      f"{(~check.flag.isin(['match', 'differs'])).sum()} informational")
print("\nrows flagged 'differs':")
print(check[check.flag == "differs"].to_string(index=False))

                                location                                                                  quantity main.tex computed  flag
                         Sec. IV-A5 text                                         keywords compared with the record  264,586  264,586 match
                         Sec. IV-A5 text                                          documents (Table 5, All records)   52,947   52,947 match
                         Sec. IV-A5 text                                exact copy of a visible record keyword (%)     45.3     45.3 match
                         Sec. IV-A5 text                                         verbatim in title or abstract (%)     40.2     40.2 match
                         Sec. IV-A5 text                             verbatim in the remaining metadata fields (%)      0.6      0.6 match
                         Sec. IV-A5 text                           paraphrase of a record keyword, cos >= 0.70 (%)      7.3      7.3 match
                         Se